# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [17]:
class Matrix:
    def __init__(self, n_or_data, m=None):
        if isinstance(n_or_data, int) and isinstance(m, int):
            self.n = n_or_data
            self.m = m
            self.data = [[0.0 for _ in range(self.m)] for _ in range(self.n)]
            return

        if isinstance(n_or_data, list):
            if len(n_or_data) == 0:
                raise ValueError("empty matrix")

            row_len = len(n_or_data[0])
            if row_len == 0:
                raise ValueError("empty row")

            for r in n_or_data:
                if not isinstance(r, list):
                    raise TypeError("need list of lists")
                if len(r) != row_len:
                    raise ValueError("rows must match length")

            self.n = len(n_or_data)
            self.m = row_len
            self.data = [[float(x) for x in r] for r in n_or_data]
            return

        raise TypeError("Matrix needs (n,m) or list of lists")

    def shape(self):
        return (self.n, self.m)

    def __getitem__(self, key):
        if isinstance(key, tuple):
            r, c = key

            if isinstance(r, slice) or isinstance(c, slice):
                rows = range(*r.indices(self.n)) if isinstance(r, slice) else [r]
                cols = range(*c.indices(self.m)) if isinstance(c, slice) else [c]
                return Matrix([[self.data[i][j] for j in cols] for i in rows])

            return self.data[r][c]

        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = float(value)
            return
        if not isinstance(value, list) or len(value) != self.m:
            raise ValueError("row must be a list of length m")
        self.data[key] = [float(x) for x in value]
    # can't overload = so assign instead. optimal
    def assign(self, other):
        if isinstance(other, list):
            other = Matrix(other)
        if not isinstance(other, Matrix):
            raise TypeError("assign expects Matrix or list of lists")
        if self.shape() != other.shape():
            raise ValueError("size mismatch")
        for i in range(self.n):
            for j in range(self.m):
                self.data[i][j] = other.data[i][j]

In [18]:
M = Matrix(2, 3)
print("M:", M.data)

N = Matrix([[1,2,3],[4,5,6]])
print("N:", N.data)

print("N[1,2]:", N[1,2])
print("N[0][1]:", N[0][1])

N[0,1] = 99
print("N after set:", N.data)

print("slice N[0:2, 1:3]:", N[0:2, 1:3].data)

P = Matrix(2, 3)
P.assign(N)
print("P after assign(N):", P.data)

M: [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
N: [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
N[1,2]: 6.0
N[0][1]: 2.0
N after set: [[1.0, 99.0, 3.0], [4.0, 5.0, 6.0]]
slice N[0:2, 1:3]: [[99.0, 3.0], [5.0, 6.0]]
P after assign(N): [[1.0, 99.0, 3.0], [4.0, 5.0, 6.0]]


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [19]:
def _copy_2d(a):
    return [r[:] for r in a]

def to_list(M):
    return _copy_2d(M.data)

def transpose(M):
    return Matrix([[M.data[i][j] for i in range(M.n)] for j in range(M.m)])

def row(M, n):
    return Matrix([M.data[n][:]])

def column(M, n):
    return Matrix([[M.data[i][n]] for i in range(M.n)])

def block(M, n0, n1, m0, m1):
    return Matrix([r[n0:n1] for r in M.data[m0:m1]])

In [20]:
A = Matrix([[1,2,3],[4,5,6],[7,8,9]])

print("A:", to_list(A))
print("A^T:", transpose(A).data)
print("row 1:", row(A, 1).data)
print("col 2:", column(A, 2).data)
print("block cols[0:2], rows[1:3]:", block(A, 0, 2, 1, 3).data)

A: [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]
A^T: [[1.0, 4.0, 7.0], [2.0, 5.0, 8.0], [3.0, 6.0, 9.0]]
row 1: [[4.0, 5.0, 6.0]]
col 2: [[3.0], [6.0], [9.0]]
block cols[0:2], rows[1:3]: [[4.0, 5.0], [7.0, 8.0]]


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [21]:
def constant(n, m, c):
    return Matrix([[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

def eye(n):
    I = zeros(n, n)
    for i in range(n):
        I[i, i] = 1.0
    return I

In [22]:
print("constant:", constant(2, 3, 5).data)
print("zeros:", zeros(2, 2).data)
print("ones:", ones(2, 2).data)
print("eye:", eye(3).data)

constant: [[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]]
zeros: [[0.0, 0.0], [0.0, 0.0]]
ones: [[1.0, 1.0], [1.0, 1.0]]
eye: [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [23]:
def scalarmul(M, c):
    return Matrix([[M.data[i][j] * float(c) for j in range(M.m)] for i in range(M.n)])

def add(M, N):
    if M.shape() != N.shape():
        raise ValueError("size mismatch")
    return Matrix([[M.data[i][j] + N.data[i][j] for j in range(M.m)] for i in range(M.n)])

def sub(M, N):
    if M.shape() != N.shape():
        raise ValueError("size mismatch")
    return Matrix([[M.data[i][j] - N.data[i][j] for j in range(M.m)] for i in range(M.n)])

def element_mult(M, N):
    if M.shape() != N.shape():
        raise ValueError("size mismatch")
    return Matrix([[M.data[i][j] * N.data[i][j] for j in range(M.m)] for i in range(M.n)])

def mat_mult(M, N):
    if M.m != N.n:
        raise ValueError("bad dimensions")
    out = Matrix(M.n, N.m)
    for i in range(M.n):
        for j in range(N.m):
            s = 0.0
            for k in range(M.m):
                s += M.data[i][k] * N.data[k][j]
            out.data[i][j] = s
    return out

def equals(M, N):
    if M.shape() != N.shape():
        return False
    for i in range(M.n):
        for j in range(M.m):
            if M.data[i][j] != N.data[i][j]:
                return False
    return True

In [24]:
A = Matrix([[1,2],[3,4]])
B = Matrix([[5,6],[7,8]])

print("scalarmul:", scalarmul(A, 2).data)
print("add:", add(A, B).data)
print("sub:", sub(A, B).data)
print("element_mult:", element_mult(A, B).data)
print("mat_mult:", mat_mult(A, B).data)
print("equals A,A:", equals(A, A))
print("equals A,B:", equals(A, B))

scalarmul: [[2.0, 4.0], [6.0, 8.0]]
add: [[6.0, 8.0], [10.0, 12.0]]
sub: [[-4.0, -4.0], [-4.0, -4.0]]
element_mult: [[5.0, 12.0], [21.0, 32.0]]
mat_mult: [[19.0, 22.0], [43.0, 50.0]]
equals A,A: True
equals A,B: False


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [25]:
def _matrix_add(self, other): return add(self, other)
def _matrix_sub(self, other): return sub(self, other)

def _matrix_mul(self, other):
    if isinstance(other, Matrix):
        return mat_mult(self, other)
    return scalarmul(self, other)

def _matrix_rmul(self, other):
    return scalarmul(self, other)

def _matrix_eq(self, other):
    if not isinstance(other, Matrix):
        return False
    return equals(self, other)

Matrix.__add__ = _matrix_add
Matrix.__sub__ = _matrix_sub
Matrix.__mul__ = _matrix_mul
Matrix.__rmul__ = _matrix_rmul
Matrix.__eq__ = _matrix_eq

In [26]:
A = Matrix([[1,2],[3,4]])
B = Matrix([[5,6],[7,8]])

print("2*A:", (2*A).data)
print("A*2:", (A*2).data)
print("A+B:", (A+B).data)
print("A-B:", (A-B).data)
print("A*B:", (A*B).data)
print("A==A:", A==A)
print("A==B:", A==B)

2*A: [[2.0, 4.0], [6.0, 8.0]]
A*2: [[2.0, 4.0], [6.0, 8.0]]
A+B: [[6.0, 8.0], [10.0, 12.0]]
A-B: [[-4.0, -4.0], [-4.0, -4.0]]
A*B: [[19.0, 22.0], [43.0, 50.0]]
A==A: True
A==B: False


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [27]:
A = Matrix([[1,2],[3,4]])
B = Matrix([[0,1],[1,0]])
C = Matrix([[2,0],[0,2]])
I = eye(2)

print("(AB)C =", ((A*B)*C).data)
print("A(BC) =", (A*(B*C)).data)
print("Equal?", ((A*B)*C) == (A*(B*C)))
print()

print("A(B+C) =", (A*(B+C)).data)
print("AB+AC  =", ((A*B)+(A*C)).data)
print("Equal?", (A*(B+C)) == ((A*B)+(A*C)))
print()

print("AB =", (A*B).data)
print("BA =", (B*A).data)
print("AB != BA ?", (A*B) != (B*A))
print()

print("AI =", (A*I).data)
print("A  =", A.data)
print("AI == A ?", (A*I) == A)

(AB)C = [[4.0, 2.0], [8.0, 6.0]]
A(BC) = [[4.0, 2.0], [8.0, 6.0]]
Equal? True

A(B+C) = [[4.0, 5.0], [10.0, 11.0]]
AB+AC  = [[4.0, 5.0], [10.0, 11.0]]
Equal? True

AB = [[2.0, 1.0], [4.0, 3.0]]
BA = [[3.0, 4.0], [1.0, 2.0]]
AB != BA ? True

AI = [[1.0, 2.0], [3.0, 4.0]]
A  = [[1.0, 2.0], [3.0, 4.0]]
AI == A ? True
